# Lab 2.5: Human Review Workflows & Confidence Calibration

**What you'll build:** A field-level accuracy analysis system that exposes hidden failures beneath aggregate metrics — and learn why 97% accuracy can be misleading.

**Estimated time:** 20-25 minutes

| Phase | What happens | Time |
|-------|-------------|------|
| 1 | The Wrong Way — aggregate accuracy hides field-level failures | 5 min |
| 2 | The Right Way — field-level analysis reveals targeted review needs | 5 min |
| 3 | Your Turn — build a stratified review routing system | 10 min |
| 4 | Stress Test — verify your routing logic across document types | 5 min |

In [ ]:
import anthropic
import json
from dotenv import load_dotenv

load_dotenv()

client = anthropic.Anthropic()
MODEL = "claude-sonnet-4-20250514"

## The Setup

You run a document extraction system that processes invoices, receipts, and insurance claims. It extracts multiple fields from each document. Your aggregate accuracy looks great — but finance keeps finding errors in specific fields.

The challenge: aggregate accuracy masks per-field failures. You need to drill down to find where the real problems are and route only the failing fields to human review.

In [ ]:
# Simulated extraction results with field-level accuracy
EXTRACTION_RESULTS = {
    "typed_invoices": {
        "volume": 8000,
        "fields": {
            "vendor_name": {"accuracy": 0.99, "confidence_avg": 0.95},
            "invoice_date": {"accuracy": 0.97, "confidence_avg": 0.92},
            "line_items": {"accuracy": 0.94, "confidence_avg": 0.88},
            "subtotal": {"accuracy": 0.96, "confidence_avg": 0.91},
            "tax_amount": {"accuracy": 0.93, "confidence_avg": 0.85},
            "total_amount": {"accuracy": 0.95, "confidence_avg": 0.90}
        }
    },
    "scanned_receipts": {
        "volume": 1500,
        "fields": {
            "vendor_name": {"accuracy": 0.88, "confidence_avg": 0.78},
            "receipt_date": {"accuracy": 0.72, "confidence_avg": 0.60},
            "line_items": {"accuracy": 0.55, "confidence_avg": 0.45},
            "subtotal": {"accuracy": 0.68, "confidence_avg": 0.55},
            "tax_amount": {"accuracy": 0.52, "confidence_avg": 0.40},
            "total_amount": {"accuracy": 0.71, "confidence_avg": 0.58}
        }
    },
    "handwritten_notes": {
        "volume": 500,
        "fields": {
            "author_name": {"accuracy": 0.65, "confidence_avg": 0.50},
            "date": {"accuracy": 0.45, "confidence_avg": 0.35},
            "amounts": {"accuracy": 0.38, "confidence_avg": 0.28},
            "description": {"accuracy": 0.70, "confidence_avg": 0.55},
            "category": {"accuracy": 0.60, "confidence_avg": 0.48},
            "reference_numbers": {"accuracy": 0.32, "confidence_avg": 0.22}
        }
    }
}

# Calculate aggregate accuracy
total_correct = 0
total_fields = 0
for doc_type, data in EXTRACTION_RESULTS.items():
    for field, stats in data["fields"].items():
        total_correct += stats["accuracy"] * data["volume"]
        total_fields += data["volume"]

aggregate_accuracy = total_correct / total_fields

print(f"=== AGGREGATE ACCURACY: {aggregate_accuracy:.1%} ===")
print(f"\nLooks great, right? Let's dig deeper...")
print(f"\nDocument types:")
for doc_type, data in EXTRACTION_RESULTS.items():
    avg_acc = sum(f["accuracy"] for f in data["fields"].values()) / len(data["fields"])
    print(f"  {doc_type}: volume={data['volume']}, avg field accuracy={avg_acc:.1%}")

---

## Phase 1: The Wrong Approach

Report aggregate accuracy and treat all documents the same way. This masks critical per-field failures.

In [ ]:
NAIVE_ANALYSIS_PROMPT = f"""You are a QA analyst reviewing an extraction system's performance.

System performance: {aggregate_accuracy:.1%} aggregate accuracy across {total_fields} field extractions.

The system processes invoices, receipts, and handwritten notes.

Question: Is this system ready for production? What is your recommendation?

Respond as JSON: {{"recommendation": "approve" or "needs_work", "reasoning": "...", "confidence": "high/medium/low"}}"""

response = client.messages.create(
    model=MODEL,
    max_tokens=300,
    messages=[{"role": "user", "content": NAIVE_ANALYSIS_PROMPT}]
)

print("=== NAIVE ANALYSIS (aggregate only) ===")
print(response.content[0].text)
print()
print("THE PROBLEM:")
print("  - Handwritten note reference_numbers: 32% accuracy")
print("  - Handwritten note amounts: 38% accuracy")
print("  - Scanned receipt tax_amount: 52% accuracy")
print("  - These are hidden behind the aggregate 90%+ number")

### Why aggregate accuracy is misleading

- **Volume weighting hides rare types.** 8,000 typed invoices at 95%+ dominate the average. 500 handwritten notes at 50% barely register.
- **Field averaging hides specific failures.** A document type with 5 fields at 95% and 1 field at 30% averages to 84% — but that 30% field may be the most important.
- **Business impact is not uniform.** A 5% error on vendor_name is annoying; a 50% error on tax_amount is a compliance risk.

---

## Phase 2: The Right Approach

Break down accuracy by document type and field. Identify the specific failures and route only failing fields to human review.

In [ ]:
DETAILED_ANALYSIS_PROMPT = f"""You are a QA analyst reviewing an extraction system's performance.

Here is the DETAILED breakdown by document type and field:

{json.dumps(EXTRACTION_RESULTS, indent=2)}

ANALYSIS INSTRUCTIONS:
1. For each document type, identify fields below 80% accuracy as FAILING.
2. For each document type, identify fields between 80-90% accuracy as AT RISK.
3. Recommend which fields need human review routing.
4. Recommend which document types need special handling.
5. Do NOT report only the aggregate accuracy — it masks failures.

Respond as JSON: {{
  "failing_fields": [{{"doc_type": "...", "field": "...", "accuracy": ...}}],
  "at_risk_fields": [{{"doc_type": "...", "field": "...", "accuracy": ...}}],
  "review_routing": {{"human_review_required": [...], "auto_approve": [...]}},
  "recommendation": "..."
}}"""

response = client.messages.create(
    model=MODEL,
    max_tokens=800,
    messages=[{"role": "user", "content": DETAILED_ANALYSIS_PROMPT}]
)

print("=== DETAILED ANALYSIS (per-field) ===")
print(response.content[0].text)

In [ ]:
# Show the full field-level breakdown
print("=" * 70)
print("FIELD-LEVEL ACCURACY BREAKDOWN")
print("=" * 70)

for doc_type, data in EXTRACTION_RESULTS.items():
    print(f"\n{doc_type.upper()} (volume: {data['volume']})")
    print(f"  {'Field':<25} {'Accuracy':>10} {'Confidence':>12} {'Status':>10}")
    print(f"  {'-'*25} {'-'*10} {'-'*12} {'-'*10}")
    for field, stats in data["fields"].items():
        acc = stats["accuracy"]
        conf = stats["confidence_avg"]
        if acc >= 0.90:
            status = "OK"
        elif acc >= 0.80:
            status = "AT RISK"
        else:
            status = "FAILING"
        print(f"  {field:<25} {acc:>9.0%} {conf:>11.0%} {status:>10}")

---

## Phase 3: Your Turn

Build a review routing system that uses field-level confidence to decide which extractions need human review.

In [ ]:
# Simulated extraction outputs with per-field confidence
SAMPLE_EXTRACTIONS = [
    {
        "doc_id": "DOC-001",
        "doc_type": "typed_invoice",
        "fields": {
            "vendor_name": {"value": "Acme Corp", "confidence": 0.97},
            "invoice_date": {"value": "2024-11-15", "confidence": 0.94},
            "total_amount": {"value": "$1,234.56", "confidence": 0.92},
            "tax_amount": {"value": "$98.77", "confidence": 0.88}
        },
        "expected_routing": "auto_approve"
    },
    {
        "doc_id": "DOC-002",
        "doc_type": "scanned_receipt",
        "fields": {
            "vendor_name": {"value": "Joe's Diner", "confidence": 0.82},
            "receipt_date": {"value": "2024-11-10", "confidence": 0.55},
            "total_amount": {"value": "$45.80", "confidence": 0.40},
            "tax_amount": {"value": "$3.66", "confidence": 0.32}
        },
        "expected_routing": "human_review",
        "expected_review_fields": ["receipt_date", "total_amount", "tax_amount"]
    },
    {
        "doc_id": "DOC-003",
        "doc_type": "typed_invoice",
        "fields": {
            "vendor_name": {"value": "TechSupplies Inc", "confidence": 0.96},
            "invoice_date": {"value": "2024-10-28", "confidence": 0.91},
            "total_amount": {"value": "$8,750.00", "confidence": 0.35},
            "tax_amount": {"value": "$700.00", "confidence": 0.89}
        },
        "expected_routing": "human_review",
        "expected_review_fields": ["total_amount"]
    },
    {
        "doc_id": "DOC-004",
        "doc_type": "handwritten_note",
        "fields": {
            "author_name": {"value": "J. Smith", "confidence": 0.45},
            "date": {"value": "Nov 2024", "confidence": 0.30},
            "amounts": {"value": "$500", "confidence": 0.25},
            "description": {"value": "Office supplies reimbursement", "confidence": 0.60}
        },
        "expected_routing": "human_review",
        "expected_review_fields": ["author_name", "date", "amounts"]
    },
    {
        "doc_id": "DOC-005",
        "doc_type": "typed_invoice",
        "fields": {
            "vendor_name": {"value": "CloudServices LLC", "confidence": 0.99},
            "invoice_date": {"value": "2024-12-01", "confidence": 0.98},
            "total_amount": {"value": "$299.99", "confidence": 0.97},
            "tax_amount": {"value": "$24.00", "confidence": 0.95}
        },
        "expected_routing": "auto_approve"
    }
]

print(f"Loaded {len(SAMPLE_EXTRACTIONS)} sample extractions")
for ext in SAMPLE_EXTRACTIONS:
    print(f"  {ext['doc_id']} ({ext['doc_type']}): expected={ext['expected_routing']}")

In [ ]:
# =============================================================
# TODO: Write a review routing prompt
# =============================================================
#
# Requirements:
# - Must use field-level confidence to make routing decisions
# - Fields below 70% confidence should be flagged for human review
# - If ANY field is flagged, the document goes to human review
# - The response must specify WHICH fields need review (not the whole doc)
#
# Think about:
# - What confidence threshold separates auto-approve from human review?
# - Should the threshold vary by field importance (amount vs name)?
# - How do you report which specific fields need review?

def route_extraction(extraction):
    """Route an extraction to human review or auto-approve."""
    prompt = f"""You are a document review routing system.

# TODO: Write your routing instructions here.
# Use field-level confidence to decide routing.
# Fields below 70% confidence need human review.

Document extraction:
{json.dumps(extraction['fields'], indent=2)}

Document type: {extraction['doc_type']}

Respond as JSON: {{{{
  "routing": "auto_approve" or "human_review",
  "review_fields": ["list of fields needing review"],
  "reasoning": "brief explanation"
}}}}"""
    
    response = client.messages.create(
        model=MODEL,
        max_tokens=300,
        messages=[{"role": "user", "content": prompt}]
    )
    
    raw = response.content[0].text.strip()
    if raw.startswith("```"):
        raw = raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {"routing": "unknown", "review_fields": [], "reasoning": raw}


# Run routing on all samples
print("=== YOUR ROUTING RESULTS ===")
routing_results = []
for ext in SAMPLE_EXTRACTIONS:
    result = route_extraction(ext)
    expected = ext["expected_routing"]
    actual = result.get("routing", "unknown")
    match = actual == expected
    routing_results.append({"doc_id": ext["doc_id"], "expected": expected, "actual": actual, "match": match, "result": result})
    tag = "CORRECT" if match else "WRONG"
    print(f"\n  {ext['doc_id']}: {actual} (expected {expected}) — {tag}")
    if result.get("review_fields"):
        print(f"    Review fields: {result['review_fields']}")

In [ ]:
# =============================================================
# Checker: validates your routing
# =============================================================

routing_accuracy = sum(1 for r in routing_results if r["match"]) / len(routing_results)

print("=== YOUR SCORECARD ===")
print(f"  Routing accuracy: {routing_accuracy:.0%}")

for r in routing_results:
    status = "PASS" if r["match"] else "FAIL"
    print(f"  {r['doc_id']}: {status} (said {r['actual']}, expected {r['expected']})")

if routing_accuracy == 1.0:
    print("\n  PASSED — all documents correctly routed!")
else:
    wrong = [r for r in routing_results if not r["match"]]
    print(f"\n  {len(wrong)} incorrect routing(s). Check your confidence threshold.")

---

## Phase 4: Stress Test

Run your routing multiple times to verify consistency.

In [ ]:
print("Running routing 3 times across all documents...\n")

all_correct = True
for run in range(3):
    run_results = []
    for ext in SAMPLE_EXTRACTIONS:
        result = route_extraction(ext)
        match = result.get("routing") == ext["expected_routing"]
        run_results.append(match)
    
    accuracy = sum(run_results) / len(run_results)
    wrong_docs = [SAMPLE_EXTRACTIONS[i]["doc_id"] for i, m in enumerate(run_results) if not m]
    print(f"  Run {run+1}: {accuracy:.0%}", end="")
    if wrong_docs:
        print(f" (wrong: {wrong_docs})")
        all_correct = False
    else:
        print(" (perfect)")

print(f"\n=== CONSISTENCY ===")
if all_correct:
    print("Perfect consistency — your routing logic produces correct decisions every time.")
else:
    print("Inconsistent — check your confidence threshold and routing logic.")

### Key Takeaways

1. **Aggregate accuracy masks field-level failures.** 97% overall can hide 32% accuracy on specific fields.
2. **Break down by document type AND field.** Per-document-type accuracy is better than aggregate, but per-field is actionable.
3. **Stratified sampling catches rare failures.** Over-represent hard document types in validation sets.
4. **Field-level confidence enables targeted review.** Route low-confidence fields (not entire documents) to humans.
5. **Validation sets must represent hard cases.** 100 clean typed invoices tells you nothing about scanned receipts.